In [1]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import SimpleRNN
import csv

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bagavathypriya/spam-ham-dataset")

print("Path to dataset files:", path)

100%|██████████| 207k/207k [00:00<00:00, 64.9MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/bagavathypriya/spam-ham-dataset/versions/1


In [3]:
file_path = f'{path}/spamhamdata.csv'
df = pd.read_csv(file_path, sep='\t', engine='python', on_bad_lines='skip', header=None, names=['label', 'message'])
print(df.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5562 entries, 0 to 5561
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5562 non-null   object
 1   message  5562 non-null   object
dtypes: object(2)
memory usage: 87.0+ KB


In [5]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

In [6]:
import string
from sklearn.preprocessing import LabelEncoder
import warnings
import nltk
nltk.download('stopwords') # Download NLTK stopwords
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.metrics.pairwise import cosine_similarity
stop=set(stopwords.words('english'))
warnings.filterwarnings("ignore")
from sklearn.feature_extraction.text import CountVectorizer
import string

stop = set(stopwords.words('english'))

def remove_num(txt):
    new = ''
    for i in str(txt): # Added str() to handle potential non-string data
        if not i.isdigit():
            new = new + i
    return new

def remove_pun(txt):
    # Fixed: The first two arguments should be empty strings, not ' '
    return str(txt).translate(str.maketrans('', '', string.punctuation))

def remove_stopwords(txt):
    word = str(txt).split()
    text = []
    for i in word:
        if i not in stop:
            text.append(i)
    # FIX: You were returning 'txt' (original). You must return " ".join(text)
    return " ".join(text)

def remove_emoji(txt):
    e = ""
    for i in str(txt):
        if i.isascii():
            e = e + i
    return e

def to_lower(txt):
    return str(txt).lower()

import pandas as pd
!pip install langdetect
!pip install deep_translator
from langdetect import detect
from deep_translator import GoogleTranslator

def translate_if_hindi(text):
    """
    Detects if the text is Hindi. If yes, translates to English.
    Otherwise, returns the original text.
    """
    # 1. Handle empty cells or missing data (NaN)
    if pd.isna(text) or not str(text).strip():
        return text

    # Convert to string just in case
    text = str(text)

    try:
        # 2. Detect the language
        # 'hi' is the standard language code for Hindi
        lang = detect(text)

        # 3. Translate if it is Hindi
        if lang == 'hi':
            # Translate from Hindi to English
            translated = GoogleTranslator(source='hi', target='en').translate(text)
            return translated
        else:
            # If it's English, Spanish, etc., leave it alone
            return text

    except Exception as e:
        # This catches errors (e.g., if the text is just numbers or emojis)
        # It will just return the original text without crashing
        return text

def master_clean(txt):
    # If the input is Null/NaN, return an empty string to avoid crashes
    if not isinstance(txt, str) and not isinstance(txt, (int, float)):
        return ""

    txt = to_lower(txt)
    txt = remove_emoji(txt)
    txt = remove_pun(txt)
    txt = remove_num(txt)
    txt = remove_stopwords(txt)
    txt = translate_if_hindi(txt)
    return txt

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 21.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=50462afd1fbc1a22d570b7f9b6c4e1a7f795f588736fa05be81dcf8b5c8cc55e
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.8 MB/s eta 0:00:00


In [7]:
df['message'] = df['message'].apply(master_clean)

In [8]:
df.head()

,label,message
0,0,go jurong point crazy available bugis n great ...
1,0,ok lar joking wif u oni
2,1,free entry wkly comp win fa cup final tkts st ...
3,0,u dun say early hor u c already say
4,0,nah dont think goes usf lives around though


In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
tokenizer = Tokenizer(oov_token="<unk>")
tokenizer.fit_on_texts(df['message'])
MAX_LEN = max([len(message.split()) for message in df['message']])

def preprocess_messages_for_rnn(messages):
    cleaned_messages = [master_clean(msg) for msg in messages]
    sequences = tokenizer.texts_to_sequences(cleaned_messages)
    padded_sequences = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

    return padded_sequences

print(f"Tokenizer vocabulary size: {len(tokenizer.word_index)}")
print(f"Maximum sequence length found in data: {MAX_LEN} words")


Tokenizer vocabulary size: 8474
Maximum sequence length found in data: 80 words


In [12]:
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 128

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=MAX_LEN),
    tf.keras.layers.SimpleRNN(units=64, return_sequences=False),
    tf.keras.layers.Dense(units=32, activation='relu'),
    tf.keras.layers.Dense(units=1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [13]:
HIST = model.fit(preprocess_messages_for_rnn(df['message']), df['label'].values, epochs=10, batch_size=32, validation_split=0.2)

Epoch 1/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step - accuracy: 0.8647 - loss: 0.4064 - val_accuracy: 0.8697 - val_loss: 0.3873
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 51ms/step - accuracy: 0.8647 - loss: 0.3982 - val_accuracy: 0.8697 - val_loss: 0.3871
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.8647 - loss: 0.3990 - val_accuracy: 0.8697 - val_loss: 0.3890
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 5s 38ms/step - accuracy: 0.8647 - loss: 0.3978 - val_accuracy: 0.8697 - val_loss: 0.3906
Epoch 5/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 0.8647 - loss: 0.3993 - val_accuracy: 0.8697 - val_loss: 0.3878
Epoch 6/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 0.8647 - loss: 0.3980 - val_accuracy: 0.8697 - val_loss: 0.3884
Epoch 7/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step - accuracy: 0.8647 - loss: 0.3982 - val_accuracy: 0.8697 - val_loss: 0.4118
Epoch 8/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.8647 - loss: 0.3983 - val_accu

In [14]:
model.save("RNN.h5")

In [16]:
import gradio as gr
from tensorflow.keras.models import load_model

# Load the saved model
loaded_model = load_model('RNN.h5')

def predict_message_type(message):
    processed_message = preprocess_messages_for_rnn([message])

    # Make prediction using the loaded model
    prediction = loaded_model.predict(processed_message)[0][0]

    # Determine if it's spam or ham based on a threshold (e.g., 0.5)
    if prediction > 0.5:
        return f"Spam (Confidence: {prediction:.2f})"
    else:
        return f"Ham (Confidence: {1 - prediction:.2f})"

# Create the Gradio interface
iface = gr.Interface(
    fn=predict_message_type,
    inputs=gr.Textbox(lines=5, placeholder="Enter your message here..."),
    outputs="text",
    title="RNN Spam/Ham Classifier",
    description="Enter a text message to classify it as Spam or Ham using a trained RNN model."
)

# Launch the interface
iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://50f62ebf389a26c45e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
